# 02 - Pandas 数据处理基础：把原始表格变成模型能学习的数据

> 配套脚本：`02_python_ai_tools/02_pandas_basics.py`

Pandas 是 Python 里最常用的表格数据处理工具。对机器学习来说，它通常承担“把脏数据变成特征矩阵”的工作：读取数据、理解字段、筛选样本、处理缺失值、构造新特征、编码类别变量，最后输出可以交给模型的 `X` 和 `y`。

## 本 Notebook 概览

本教程不是原脚本的逐行搬运，而是围绕“机器学习数据准备”重新组织内容：

1. DataFrame / Series 的直觉与基本结构
2. 快速理解一个数据集：形状、类型、统计量、分布
3. 选择、过滤与排序：像问问题一样查询数据
4. 缺失值、异常值与重复值：真实数据的三大常见问题
5. 数据转换与特征工程：新增列、分箱、映射、apply
6. 分组聚合与透视表：从明细数据总结规律
7. 类别编码与数值标准化：让模型“看得懂”数据
8. 综合实战：构建一个可用于机器学习的特征矩阵
9. 常见误区与练习

学习建议：每个代码单元都可以单独运行、修改参数并观察输出。Pandas 最好的学习方式不是背 API，而是不断提出问题，然后用表格操作回答问题。


In [ ]:
# 环境准备
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except Exception:
    display = print

# 让 Notebook 中的图表更清晰
plt.rcParams['figure.figsize'] = (8, 4.8)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

# 尝试设置中文字体；如果系统没有这些字体，图中中文可能显示为方框，但不影响代码学习
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)

print('NumPy version:', np.__version__)
print('Pandas version:', pd.__version__)


## 1. DataFrame 的直觉：带标签的二维表

Pandas 中最核心的两个对象是：

| 对象 | 可以类比为 | 典型用途 |
|---|---|---|
| `Series` | 一列数据，带行索引 | 单个变量，例如年龄、收入、标签 |
| `DataFrame` | Excel 表 / SQL 表 | 多个变量组成的数据集 |

一个 `DataFrame` 有三个关键组成部分：

- 行索引 `index`：每一行的标签
- 列名 `columns`：每个字段的名称
- 数据值 `values`：真正参与计算的数据

机器学习中，很多时候我们会把 `DataFrame` 进一步拆成：

- 特征表 `X`：输入变量
- 标签 `y`：要预测的目标变量


In [ ]:
# 从字典创建一个小型 DataFrame
people = pd.DataFrame({
    '姓名': ['张三', '李四', '王五', '赵六', '钱七'],
    '年龄': [25, 30, 35, 28, 32],
    '身高_cm': [175, 168, 182, 170, 178],
    '体重_kg': [70, 55, 85, 65, 75],
    '城市': ['北京', '上海', '北京', '广州', '上海']
})

people


In [ ]:
print('形状 shape:', people.shape)
print('行索引 index:', people.index.tolist())
print('列名 columns:', people.columns.tolist())
print('\n数据类型 dtypes:')
print(people.dtypes)


### 1.1 Series：DataFrame 中的一列

当我们用 `df['列名']` 取出单列时，得到的是 `Series`。Series 保留了行索引，所以它不仅是一个数组，也知道每个值对应哪一行。


In [ ]:
age = people['年龄']
print(type(age))
print(age)

print('\nSeries 的均值:', age.mean())
print('Series 的最大值:', age.max())


## 2. 构造一个更接近真实任务的数据集

为了演示 Pandas 在机器学习前处理中的作用，我们构造一个“电商用户是否购买”的模拟数据集。它包含：

- 数值特征：年龄、月收入、浏览时长、历史购买次数
- 类别特征：性别、会员等级、城市、流量来源
- 目标变量：`是否购买`
- 人为加入的缺失值、异常值和重复行

这比干净的小表格更接近真实业务数据。


In [ ]:
def make_customer_dataset(n=300, seed=42):
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({
        '用户ID': [f'U{i:04d}' for i in range(1, n + 1)],
        '年龄': rng.integers(18, 66, n),
        '月收入': rng.normal(8500, 3200, n).round().astype(int),
        '网站浏览时间_分钟': rng.exponential(12, n).round(1),
        '历史购买次数': rng.poisson(3, n),
        '性别': rng.choice(['男', '女'], n),
        '会员等级': rng.choice(['普通', '银卡', '金卡'], n, p=[0.62, 0.28, 0.10]),
        '城市': rng.choice(['北京', '上海', '广州', '深圳', '成都'], n, p=[0.22, 0.22, 0.18, 0.18, 0.20]),
        '流量来源': rng.choice(['搜索', '广告', '朋友推荐', '社交媒体'], n, p=[0.38, 0.27, 0.18, 0.17]),
    })

    # 根据特征生成购买概率：这让数据中真的存在可学习的规律
    level_score = df['会员等级'].map({'普通': 0.0, '银卡': 0.45, '金卡': 0.9})
    source_score = df['流量来源'].map({'搜索': 0.1, '广告': -0.05, '朋友推荐': 0.45, '社交媒体': 0.2})
    z = (
        -2.1
        + 0.018 * (df['年龄'] - 30)
        + 0.00023 * (df['月收入'] - 8500)
        + 0.055 * df['网站浏览时间_分钟']
        + 0.16 * df['历史购买次数']
        + level_score
        + source_score
    )
    prob = 1 / (1 + np.exp(-z))
    df['是否购买'] = (rng.random(n) < prob).astype(int)

    # 人为加入缺失值
    df.loc[rng.choice(n, 16, replace=False), '月收入'] = np.nan
    df.loc[rng.choice(n, 10, replace=False), '会员等级'] = np.nan
    df.loc[rng.choice(n, 8, replace=False), '网站浏览时间_分钟'] = np.nan

    # 人为加入异常值：收入极端值、浏览时间极端值
    outlier_idx = rng.choice(n, 4, replace=False)
    df.loc[outlier_idx[:2], '月收入'] = [500, 60000]
    df.loc[outlier_idx[2:], '网站浏览时间_分钟'] = [180, 240]

    # 人为加入重复行
    duplicate_rows = df.sample(3, random_state=seed)
    df = pd.concat([df, duplicate_rows], ignore_index=True)
    return df

customers = make_customer_dataset()
customers.head()


## 3. 快速理解数据集：先看“地图”，再深入分析

面对一个陌生数据集，不要急着建模。推荐的第一轮检查是：

1. `shape`：有多少行、多少列？
2. `head()` / `tail()`：数据长什么样？
3. `info()`：每列类型是什么？缺失值有多少？
4. `describe()`：数值列范围是否合理？有没有极端值？
5. `value_counts()`：类别变量分布是否严重不平衡？


In [ ]:
print('数据形状:', customers.shape)
display(customers.head(5))
display(customers.tail(3))


In [ ]:
# info() 默认直接打印到标准输出
customers.info()


In [ ]:
# include='all' 同时查看数值列和类别列的摘要
customers.describe(include='all').T


### 3.1 分布图：比数字摘要更直观

`describe()` 能告诉我们均值、分位数，但图表更容易暴露问题。例如：

- 直方图看数值分布是否偏斜
- 箱线图看异常值
- 条形图看类别变量是否不平衡


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
customers['年龄'].hist(bins=18, ax=axes[0], color='#4C78A8')
axes[0].set_title('年龄分布')
axes[0].set_xlabel('年龄')
axes[0].set_ylabel('人数')

customers['月收入'].hist(bins=30, ax=axes[1], color='#F58518')
axes[1].set_title('月收入分布（含缺失与异常）')
axes[1].set_xlabel('月收入')

customers['网站浏览时间_分钟'].hist(bins=30, ax=axes[2], color='#54A24B')
axes[2].set_title('浏览时间分布（右偏明显）')
axes[2].set_xlabel('分钟')

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
customers['会员等级'].value_counts(dropna=False).plot(kind='bar', ax=axes[0], color='#B279A2')
axes[0].set_title('会员等级分布')
axes[0].set_xlabel('会员等级')
axes[0].set_ylabel('人数')

customers['是否购买'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color=['#E45756', '#72B7B2'])
axes[1].set_title('目标变量分布')
axes[1].set_xlabel('是否购买')
axes[1].set_xticklabels(['未购买(0)', '购买(1)'], rotation=0)
axes[1].set_ylabel('人数')

plt.tight_layout()
plt.show()


## 4. 选择、过滤与排序：用 DataFrame 回答问题

Pandas 的选择方式主要分为三类：

| 写法 | 含义 | 例子 |
|---|---|---|
| `df['列名']` | 选择单列 | `df['年龄']` |
| `df[['列1', '列2']]` | 选择多列 | `df[['年龄', '月收入']]` |
| `df.loc[行条件, 列名]` | 按标签选择 | `df.loc[df['年龄'] > 30, ['年龄']]` |
| `df.iloc[行位置, 列位置]` | 按整数位置选择 | `df.iloc[:5, :3]` |

对新手来说，最推荐熟练掌握 `loc`，因为它让“筛选哪些行”和“保留哪些列”写得很清楚。


In [ ]:
# 选择几列，查看高价值字段
customers[['用户ID', '年龄', '月收入', '会员等级', '是否购买']].head()


In [ ]:
# loc：选择“月收入超过 15000 且已购买”的用户，只展示部分列
high_income_buyers = customers.loc[
    (customers['月收入'] > 15000) & (customers['是否购买'] == 1),
    ['用户ID', '年龄', '月收入', '会员等级', '城市', '是否购买']
]

high_income_buyers.sort_values('月收入', ascending=False).head(10)


In [ ]:
# isin：筛选多个类别
first_tier = customers.loc[
    customers['城市'].isin(['北京', '上海', '深圳']),
    ['用户ID', '城市', '流量来源', '是否购买']
]

first_tier.head()


### 4.1 条件过滤的两个常见坑

1. 多个条件必须用 `&`、`|`，不能用 Python 的 `and`、`or`
2. 每个条件都要加括号，例如：`(df['年龄'] > 30) & (df['城市'] == '北京')`

原因是 Pandas 的条件判断返回的是一整列布尔值，而不是单个 `True/False`。


In [ ]:
mask = (customers['年龄'] >= 30) & (customers['年龄'] <= 45) & (customers['城市'] == '北京')
print('符合条件的行数:', mask.sum())
customers.loc[mask, ['用户ID', '年龄', '城市', '月收入', '是否购买']].head()


## 5. 缺失值、重复值与异常值：真实数据的常态

真实数据很少是干净的。常见问题包括：

- 缺失值：用户没有填写、采集失败、系统字段变更
- 重复值：数据合并或埋点重复上报
- 异常值：录入错误、极端行为、单位不一致

处理策略没有唯一答案，关键是理解业务含义，并避免把测试集信息泄露到训练过程。


In [ ]:
missing = customers.isna().sum().sort_values(ascending=False)
missing_rate = (customers.isna().mean() * 100).round(2)
missing_report = pd.DataFrame({'缺失数量': missing, '缺失比例%': missing_rate[missing.index]})
missing_report[missing_report['缺失数量'] > 0]


In [ ]:
# 缺失值热力图：每个黑色点代表一个缺失位置
plt.figure(figsize=(10, 4))
plt.imshow(customers.isna(), aspect='auto', interpolation='nearest', cmap='Greys')
plt.title('缺失值位置图：行 x 列')
plt.xlabel('列')
plt.ylabel('行号')
plt.xticks(range(customers.shape[1]), customers.columns, rotation=45, ha='right')
plt.colorbar(label='是否缺失')
plt.tight_layout()
plt.show()


In [ ]:
print('完全重复行数量:', customers.duplicated().sum())
print('用户ID 重复数量:', customers.duplicated(subset=['用户ID']).sum())

customers.loc[customers.duplicated(subset=['用户ID'], keep=False)].sort_values('用户ID').head(10)


### 5.1 处理缺失值：删除还是填充？

常见策略：

| 方法 | 适用情况 | 风险 |
|---|---|---|
| 删除行 `dropna()` | 缺失很少，且缺失随机 | 丢失样本，可能引入偏差 |
| 均值填充 | 数值列近似对称分布 | 容易受异常值影响 |
| 中位数填充 | 数值列偏斜或有异常值 | 会压缩变量分布 |
| 众数/固定值填充 | 类别列 | 可能强化多数类别 |
| 增加“是否缺失”特征 | 缺失本身可能有信息 | 特征更多，需小心解释 |

下面我们保留一份清洗副本，避免直接覆盖原始数据。


In [ ]:
clean = customers.copy()

# 1. 删除完全重复行：通常是安全的第一步
before = len(clean)
clean = clean.drop_duplicates()
print(f'删除完全重复行: {before} -> {len(clean)}')

# 2. 为关键数值列增加“是否缺失”指示列：有时缺失本身就是信号
for col in ['月收入', '网站浏览时间_分钟']:
    clean[col + '_是否缺失'] = clean[col].isna().astype(int)

# 3. 数值列用中位数填充，类别列用“未知”填充
clean['月收入'] = clean['月收入'].fillna(clean['月收入'].median())
clean['网站浏览时间_分钟'] = clean['网站浏览时间_分钟'].fillna(clean['网站浏览时间_分钟'].median())
clean['会员等级'] = clean['会员等级'].fillna('未知')

print('\n清洗后缺失值总数:', clean.isna().sum().sum())
clean.head()


### 5.2 异常值：删除、截断还是保留？

异常值不一定是错误。比如一个用户真的可能浏览 240 分钟，也可能只是网页一直开着。处理异常值时需要结合业务。

这里演示一种常见的温和方法：用分位数进行截断（winsorization），把过小/过大的值压到合理边界，而不是直接删除样本。


In [ ]:
def clip_by_quantile(s, low=0.01, high=0.99):
    lower = s.quantile(low)
    upper = s.quantile(high)
    return s.clip(lower, upper), lower, upper

for col in ['月收入', '网站浏览时间_分钟']:
    clean[col + '_截断前'] = clean[col]
    clean[col], low, high = clip_by_quantile(clean[col], 0.01, 0.99)
    print(f'{col}: 1%分位={low:.2f}, 99%分位={high:.2f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
clean[['月收入_截断前', '月收入']].plot(kind='box', ax=axes[0])
axes[0].set_title('月收入：截断前后对比')

clean[['网站浏览时间_分钟_截断前', '网站浏览时间_分钟']].plot(kind='box', ax=axes[1])
axes[1].set_title('浏览时间：截断前后对比')

plt.tight_layout()
plt.show()


## 6. 数据转换与特征工程：把原始字段变成更有用的信号

特征工程的目标不是“堆更多列”，而是把业务含义表达得更清楚。常见方法包括：

- 数值运算：比例、差值、对数变换
- 分箱：把连续值变成区间标签
- 映射：把有序类别转成分数
- 自定义函数：把复杂规则封装起来

下面从清洗后的 `clean` 数据继续。


In [ ]:
feature_df = clean.copy()

# 对右偏分布做 log1p 变换，常用于收入、浏览时长、金额等非负变量
feature_df['log_月收入'] = np.log1p(feature_df['月收入'])
feature_df['log_浏览时间'] = np.log1p(feature_df['网站浏览时间_分钟'])

# 比例/强度类特征：平均每次历史购买对应的浏览时间
feature_df['平均每次购买浏览分钟'] = feature_df['网站浏览时间_分钟'] / (feature_df['历史购买次数'] + 1)

# 有序类别映射：未知单独用 -1 表示
feature_df['会员等级分数'] = feature_df['会员等级'].map({'未知': -1, '普通': 0, '银卡': 1, '金卡': 2})

# 分箱：把年龄变成更容易解释的人群段
feature_df['年龄段'] = pd.cut(
    feature_df['年龄'],
    bins=[17, 25, 35, 45, 55, 70],
    labels=['18-25', '26-35', '36-45', '46-55', '56+']
)

feature_df[['年龄', '年龄段', '月收入', 'log_月收入', '网站浏览时间_分钟', 'log_浏览时间', '会员等级', '会员等级分数']].head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
feature_df['月收入'].hist(bins=30, ax=axes[0], color='#F58518')
axes[0].set_title('原始月收入分布')
axes[0].set_xlabel('月收入')

feature_df['log_月收入'].hist(bins=30, ax=axes[1], color='#4C78A8')
axes[1].set_title('log1p(月收入) 后的分布')
axes[1].set_xlabel('log_月收入')

plt.tight_layout()
plt.show()


### 6.1 `assign` 与链式写法

Pandas 支持链式操作，让数据处理流程更像一条管道。优点是中间变量少，缺点是太长时可读性会下降。


In [ ]:
summary_table = (
    feature_df
    .assign(购买标签=lambda d: d['是否购买'].map({0: '未购买', 1: '购买'}))
    .loc[:, ['城市', '会员等级', '年龄段', '购买标签', '月收入']]
    .sort_values(['城市', '会员等级'])
    .head(8)
)
summary_table


## 7. 分组聚合：从“明细记录”提炼“群体规律”

机器学习之前，我们经常先做探索性数据分析（EDA）。`groupby` 可以回答类似问题：

- 哪个会员等级购买率最高？
- 哪个城市平均收入最高？
- 不同流量来源的购买转化率有什么差异？

`groupby` 的思路是：先按某些列分组，再对每组做统计。


In [ ]:
level_stats = (
    feature_df
    .groupby('会员等级', observed=False)
    .agg(
        用户数=('用户ID', 'count'),
        购买率=('是否购买', 'mean'),
        平均收入=('月收入', 'mean'),
        平均浏览分钟=('网站浏览时间_分钟', 'mean')
    )
    .sort_values('购买率', ascending=False)
)

level_stats.round(3)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
level_stats['购买率'].plot(kind='bar', ax=axes[0], color='#72B7B2')
axes[0].set_title('不同会员等级的购买率')
axes[0].set_ylabel('购买率')
axes[0].tick_params(axis='x', rotation=0)

source_rate = feature_df.groupby('流量来源')['是否购买'].mean().sort_values(ascending=False)
source_rate.plot(kind='bar', ax=axes[1], color='#ECA82C')
axes[1].set_title('不同流量来源的购买率')
axes[1].set_ylabel('购买率')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()


### 7.1 透视表：二维分组更适合做对比

`pivot_table` 可以同时按行和列分组。例如：观察不同城市、不同会员等级的购买率。


In [ ]:
pivot = pd.pivot_table(
    feature_df,
    values='是否购买',
    index='城市',
    columns='会员等级',
    aggfunc='mean'
)
pivot.round(3)


In [ ]:
plt.figure(figsize=(7, 5))
plt.imshow(pivot.fillna(0), cmap='YlGnBu', aspect='auto')
plt.title('城市 x 会员等级：购买率热力图')
plt.xticks(range(len(pivot.columns)), pivot.columns)
plt.yticks(range(len(pivot.index)), pivot.index)
plt.colorbar(label='购买率')

for i, city in enumerate(pivot.index):
    for j, level in enumerate(pivot.columns):
        value = pivot.loc[city, level]
        text = 'NA' if pd.isna(value) else f'{value:.2f}'
        plt.text(j, i, text, ha='center', va='center', color='black')

plt.tight_layout()
plt.show()


## 8. 类别编码：把文字变量变成模型可用的数字

大多数机器学习模型不能直接处理“北京”“银卡”这样的字符串。常见编码方式：

| 方法 | 适用变量 | Pandas 写法 |
|---|---|---|
| One-Hot 编码 | 无大小顺序的类别，如城市、流量来源 | `pd.get_dummies()` |
| 有序编码 | 有明确顺序的类别，如普通/银卡/金卡 | `map()` |
| 频数编码 | 高基数类别，如用户所在小区 | `value_counts()` + `map()` |

注意：不要把无序类别简单编码为 0、1、2，否则模型可能误以为类别之间有大小关系。


In [ ]:
cat_cols = ['性别', '城市', '流量来源', '年龄段']
dummies = pd.get_dummies(feature_df[cat_cols], prefix=cat_cols, dtype=int)
print('One-Hot 后的列数:', dummies.shape[1])
dummies.head()


In [ ]:
# 把 One-Hot 编码结果拼回特征表的一小部分看看
encoded_preview = pd.concat([
    feature_df[['用户ID', '性别', '城市', '流量来源', '年龄段']].head(5),
    dummies.head(5)
], axis=1)
encoded_preview


## 9. 数值标准化：让不同量纲的特征更公平

很多模型对特征尺度敏感，例如线性模型、逻辑回归、KNN、神经网络。年龄可能是几十，收入可能是几万，如果不标准化，收入列可能主导模型训练。

常见标准化方式：

- Z-score 标准化：$z = \frac{x - \mu}{\sigma}$，均值约为 0，标准差约为 1
- Min-Max 归一化：缩放到 `[0, 1]`

这里手动实现 Z-score，有助于理解背后的数学。


In [ ]:
numeric_cols = [
    '年龄', '月收入', '网站浏览时间_分钟', '历史购买次数',
    'log_月收入', 'log_浏览时间', '平均每次购买浏览分钟',
    '会员等级分数', '月收入_是否缺失', '网站浏览时间_分钟_是否缺失'
]

scaled_numeric = feature_df[numeric_cols].copy()
means = scaled_numeric.mean()
stds = scaled_numeric.std(ddof=0).replace(0, 1)
scaled_numeric = (scaled_numeric - means) / stds

scaled_numeric.describe().loc[['mean', 'std']].round(3)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
feature_df[['年龄', '月收入', '网站浏览时间_分钟']].plot(kind='box', ax=axes[0])
axes[0].set_title('标准化前：量纲差异很大')
axes[0].tick_params(axis='x', rotation=20)

scaled_numeric[['年龄', '月收入', '网站浏览时间_分钟']].plot(kind='box', ax=axes[1])
axes[1].set_title('Z-score 标准化后：尺度接近')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()


## 10. 综合实战：准备机器学习特征矩阵

现在把前面的步骤串起来：

1. 从原始表复制数据
2. 删除重复行
3. 分离标签 `y`
4. 处理缺失值、异常值
5. 构造新特征
6. One-Hot 编码类别变量
7. 标准化数值变量
8. 划分训练集和测试集

为了教学清晰，我们只用 Pandas 和 NumPy 手写流程；在真实项目中也可以使用 scikit-learn 的 `Pipeline` 和 `ColumnTransformer`。


In [ ]:
def train_test_split_numpy(X, y, test_ratio=0.2, seed=42):
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(X))
    test_size = int(len(X) * test_ratio)
    test_idx = indices[:test_size]
    train_idx = indices[test_size:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


def prepare_features(raw_df):
    df = raw_df.drop_duplicates().copy()

    # 标签
    y = df['是否购买'].to_numpy(dtype=np.int64)

    # 缺失指示
    for col in ['月收入', '网站浏览时间_分钟']:
        df[col + '_是否缺失'] = df[col].isna().astype(int)

    # 填充缺失
    df['月收入'] = df['月收入'].fillna(df['月收入'].median())
    df['网站浏览时间_分钟'] = df['网站浏览时间_分钟'].fillna(df['网站浏览时间_分钟'].median())
    df['会员等级'] = df['会员等级'].fillna('未知')

    # 温和截断异常值
    for col in ['月收入', '网站浏览时间_分钟']:
        lo, hi = df[col].quantile([0.01, 0.99])
        df[col] = df[col].clip(lo, hi)

    # 特征工程
    df['log_月收入'] = np.log1p(df['月收入'])
    df['log_浏览时间'] = np.log1p(df['网站浏览时间_分钟'])
    df['平均每次购买浏览分钟'] = df['网站浏览时间_分钟'] / (df['历史购买次数'] + 1)
    df['会员等级分数'] = df['会员等级'].map({'未知': -1, '普通': 0, '银卡': 1, '金卡': 2})
    df['年龄段'] = pd.cut(df['年龄'], bins=[17, 25, 35, 45, 55, 70], labels=['18-25', '26-35', '36-45', '46-55', '56+'])

    numeric_cols = [
        '年龄', '月收入', '网站浏览时间_分钟', '历史购买次数',
        'log_月收入', 'log_浏览时间', '平均每次购买浏览分钟',
        '会员等级分数', '月收入_是否缺失', '网站浏览时间_分钟_是否缺失'
    ]
    categorical_cols = ['性别', '城市', '流量来源', '年龄段']

    X_num = df[numeric_cols].copy()
    X_num = (X_num - X_num.mean()) / X_num.std(ddof=0).replace(0, 1)

    X_cat = pd.get_dummies(df[categorical_cols], dtype=int)
    X_df = pd.concat([X_num, X_cat], axis=1)

    return X_df, y

X_df, y = prepare_features(customers)
X = X_df.to_numpy(dtype=np.float64)
X_train, X_test, y_train, y_test = train_test_split_numpy(X, y, test_ratio=0.2, seed=42)

print('特征列数:', X_df.shape[1])
print('特征矩阵 X:', X.shape)
print('标签 y:', y.shape)
print('训练集:', X_train.shape, y_train.shape)
print('测试集:', X_test.shape, y_test.shape)
print('训练集购买率:', f'{y_train.mean():.2%}')
print('测试集购买率:', f'{y_test.mean():.2%}')

X_df.head()


### 10.1 用一个简单基线验证数据是否可用

即使本节重点不是建模，我们也可以写一个非常简单的规则模型，检查数据准备是否合理。

这里构造一个“倾向分数”：浏览时间越长、历史购买次数越多、会员等级越高，越可能购买。然后用训练集上的中位数作为阈值进行预测。

这不是严肃模型，只是一个 sanity check：如果数据处理流程出错，通常在这一步会暴露出来。


In [ ]:
# 用 DataFrame 列名定位几个标准化后的特征
score_cols = ['网站浏览时间_分钟', '历史购买次数', '会员等级分数', 'log_月收入']
score = X_df[score_cols].sum(axis=1).to_numpy()

score_train, score_test, y_train2, y_test2 = train_test_split_numpy(score.reshape(-1, 1), y, test_ratio=0.2, seed=42)
threshold = np.median(score_train)
y_pred = (score_test.ravel() >= threshold).astype(int)
accuracy = (y_pred == y_test2).mean()

print('简单规则模型准确率:', f'{accuracy:.2%}')
print('基线：总是预测多数类的准确率:', f'{max(y_test2.mean(), 1 - y_test2.mean()):.2%}')


In [ ]:
# 混淆矩阵（不依赖 sklearn）
conf = pd.crosstab(
    pd.Series(y_test2, name='真实值'),
    pd.Series(y_pred, name='预测值'),
    rownames=['真实值'], colnames=['预测值']
)
conf


## 11. 保存与加载：CSV 是最常见的交换格式

Pandas 可以读写很多格式，例如 CSV、Excel、Parquet、JSON、SQL。初学阶段最常用的是 CSV。

注意：

- `index=False` 通常可以避免把行号也保存成一列
- 读取中文 CSV 时，如果遇到乱码，可能需要指定 `encoding='utf-8-sig'` 或 `encoding='gbk'`
- 真实大数据集更推荐 Parquet，因为类型保存更好、读写更快


In [ ]:
# 保存一个小样本到临时 CSV，再读回来
sample_path = '02_python_ai_tools/pandas_customer_sample.csv'
feature_df.head(20).to_csv(sample_path, index=False, encoding='utf-8-sig')
loaded_sample = pd.read_csv(sample_path)

print('已保存:', sample_path)
print('读取后的形状:', loaded_sample.shape)
loaded_sample.head(3)


## 12. 常见误区与检查清单

### 常见误区

1. **直接覆盖原始数据**：建议保留 `raw_df`，清洗过程用副本。
2. **在划分训练/测试集之前用全量数据计算统计量**：严格建模时，均值、中位数、标准差应该只从训练集估计，再应用到测试集，避免数据泄露。
3. **把无序类别编码成 0/1/2**：模型可能误解大小关系，城市这类变量应使用 One-Hot。
4. **只看均值不看分布**：均值可能被异常值严重影响，图表和分位数同样重要。
5. **缺失值一律删除**：如果缺失不是随机的，删除会引入偏差；缺失指示列有时很有价值。
6. **忽略目标变量分布**：分类任务中正负样本极度不平衡时，准确率可能误导判断。

### 数据准备检查清单

- [ ] 行数、列数是否符合预期？
- [ ] 每列数据类型是否正确？数值是否被读成字符串？
- [ ] 缺失值比例是否可接受？处理策略是否合理？
- [ ] 是否有重复样本或重复 ID？
- [ ] 数值列是否存在明显异常值或单位问题？
- [ ] 类别列是否有拼写不一致或未知类别？
- [ ] 特征矩阵是否全是数值？是否还包含字符串？
- [ ] 训练集和测试集是否分开？是否避免了数据泄露？


## 13. 本节总结与练习

### 关键要点

- `DataFrame` 是机器学习数据准备的核心容器。
- EDA 的第一步是看结构、类型、缺失、分布，而不是马上训练模型。
- `loc`、布尔过滤、`groupby`、`pivot_table` 是 Pandas 的高频操作。
- 缺失值、异常值、重复值需要结合业务含义处理。
- 类别变量要编码，数值变量常需要标准化。
- 最终目标是得到形如 `(n_samples, n_features)` 的数值特征矩阵 `X` 和标签 `y`。

### 与 AI 的连接

Pandas 处在 AI 工作流的最前面：

`原始数据 -> Pandas 清洗/特征工程 -> NumPy/模型输入 -> 训练 -> 评估`

模型效果不好时，问题往往不在模型结构，而在数据质量、特征表达和评估方式。

### 练习

1. 把 `年龄段` 的分箱边界改成 `[17, 30, 45, 60, 70]`，观察购买率是否变化。
2. 尝试用 `城市` 和 `流量来源` 做二维透视表，比较不同组合的购买率。
3. 把缺失值处理从“中位数填充”改成“均值填充”，比较数值分布变化。
4. 增加一个新特征：`收入每分钟浏览 = 月收入 / (网站浏览时间_分钟 + 1)`，观察它与购买率的关系。
5. 严格模拟真实建模流程：先划分训练/测试集，再只用训练集统计量填充和标准化测试集。
